# Задание №1

## 1.1

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score

In [ ]:
df = pd.read_csv('Salary.csv')

In [ ]:
df.head()

,YearsExperience,Salary
0,1.1,39343
1,1.3,46205
2,1.5,37731
3,2.0,43525
4,2.2,39891


In [ ]:
df.describe()

,YearsExperience,Salary
count,30.000000,30.000000
mean,5.313333,76003.000000
std,2.837888,27414.429785
min,1.100000,37731.000000
25%,3.200000,56720.750000
50%,4.700000,65237.000000
75%,7.700000,100544.750000
max,10.500000,122391.000000


## 1.2

### Воспользуемся plotly

In [ ]:
hist_salary = px.histogram(df, x='Salary', nbins=5, title='Гистограмма распределения зарплат') #делаем шаг в 20к, для этого нужно пять интервалов
hist_salary.update_layout(bargap=0.1) #отделяем столбцы друг от друга`
hist_salary.show()

NameError: name 'px' is not defined

In [ ]:
hist_experience = px.histogram(df, x='YearsExperience', nbins=15, title='Гистограмма распределения опыта работы')
hist_experience.update_layout(bargap=0.1,xaxis=dict(tickmode='linear',dtick=1)) #отделяем столбцы друг от друга и делаем шаг по оси х равным 1 году
hist_experience.show()

## 1.3

In [ ]:
sct_plt = px.scatter(df, x='YearsExperience', y='Salary', title='Зависимость зарплаты от опыта работы', size='Salary', size_max=20)
sct_plt.show()

Зарплата имеет линейную зависимость от опыта работы – чем больше опыт, тем выше зарплата

## 1.4

In [ ]:
scaler = MinMaxScaler()
df['Scalared_Salary']= scaler.fit_transform(df[['Salary']]) #отскалировали зп

Определим признаки и разделим на тестовую и обучающую выборки

In [ ]:
X = df[['YearsExperience']] #признак
y = df[['Scalared_Salary']] #целевая переменная

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = 0.3, random_state = 68)

Я пробовал разные соотношения тестовой и обучающей выборки, а также разные random_state, в итоге сложно подобрать оптимальный размер тестовой выборки, поскольку данных мало и результат сильно зависит от того, как именно данные поделятся по выборкам, но в целом 30% на тестовую выборку дали хорошие результаты

## 1.5

In [ ]:
model = LinearRegression() #инициализировали модель
model.fit(X_train,y_train) #обучили модель

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


Посмотрим MAE на обучающей и тестовой выборке

In [ ]:
test_predict = model.predict(X_test) # Считаем выходы по тестовой выборке
train_predict = model.predict(X_train) # Считаем выходы по тренировочной выборке

In [ ]:
MAE_Test = mean_absolute_error(test_predict, y_test)
MAE_Train = mean_absolute_error(train_predict, y_train)
print(f"MAE Test = {round(MAE_Test,3)}")
print(f"MAE Train = {round(MAE_Train,3)}")

MAE Test = 0.047
MAE Train = 0.058


На тестовой выборке модель работает чуть лучше, что вполне может быть для такого маленького количества данных, но ошибка все равно мала, учитывая, что отскалированная ЗП находится в промежутке [0,1]. Использовать MSE в данном случае будет не так эффективно, поскольку данные от 0 до 1 и MSE будет очень маленьким

Также оценим то, как мы покрываем разброс в данных с помощью R2

In [ ]:
r2_Test = r2_score(y_test,test_predict)
r2_Train= r2_score(y_train, train_predict)
print(f"R2 Test = {round(r2_Test,2)}")
print(f"R2 Train = {round(r2_Train,2)}")

R2 Test = 0.98
R2 Train = 0.93


R2 На тестовых данных чуть лучше, чем на тренировочных, но у обоих показатель очень близок к еднице, а значит модель хорошо описывает разброс данных

## 1.6

Коэффиценты модели отскалированные, если строить прямую по этим данным, то регрессия будет показывать зависимость только на отскалированных данных (от 0 до 1) (возможно, это будет полезно, если новые данные будут приходить уже отскалированные, но скорее всего это будет не так), чтобы построить график на изначальных данных, их нужно отскалировать обратно

In [ ]:
#Коэффиценты в скалированном пространстве
w0_scaled = model.intercept_[0]
w1_scaled = model.coef_[0][0]
print(f"w0 = {round(w0_scaled,2)}")
print(f"w1 = {round(w1_scaled,2)}")


w0 = -0.15
w1 = 0.11


### График на скалированных данных (зп от 0 до 1)

In [ ]:
x_value = np.linspace(np.min(df['YearsExperience']), np.max(df['YearsExperience']), 100)
#создали массив из 100 чисел от мин до макс опыта работы
y_value_scaled = w0_scaled + w1_scaled * x_value
# скалированный y_value
sct_plt = px.scatter(df, x='YearsExperience', y='Scalared_Salary', title='Зависимость зарплаты от опыта работы')
#делаем scatter plot
sct_plt.add_trace(go.Scatter(x=x_value, y=y_value_scaled, mode='lines', name='Линия регрессии', line=dict(color='red', width=5)))
#делаем линию регрессии на scatter plot
sct_plt.show()

### График на изначальных данных

Трансформируем данные обратно с помощью **scaler.inverse_transform**

In [ ]:
x_value = np.linspace(np.min(df['YearsExperience']), np.max(df['YearsExperience']), 100)
#массив из 100 чисел от мин до макс опыта работы
y_value_scaled = w0_scaled + w1_scaled * x_value
# скалированный y value
y_value_original = scaler.inverse_transform(y_value_scaled.reshape(-1, 1)).flatten()
# трансформируем обратно
sct_plt = px.scatter(df, x='YearsExperience', y='Salary', title='Зависимость зарплаты от опыта работы')
sct_plt.add_trace(go.Scatter(x=x_value, y=y_value_original, mode='lines', name='Линия регрессии', line=dict(color='red', width=5)))
sct_plt.show()

# Задание 2

## 2.1

In [ ]:
df1 = pd.read_csv('insurance.csv')

## 2.2

Посмотрим сначала что находится внутри датасета

In [ ]:
df1.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Очистим датасет от дубликатов перед началом работы, так как модель может придавать дополнительную важность дубликатам, или переобучаться на повторяющихся примерах, если они попадут в тестовую и обучающую выборки одновременно (она просто запомнит их)

In [ ]:
df1 = df1.drop_duplicates()

In [ ]:
df1.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Для иллюстраций будет использован plotly

Поскольку возраст распределен от 18 до 64, то есть 47 "возрастов", поскольку наблюдений много, можно спокойно сделать один столбец = один год

In [ ]:
hist_age = px.histogram(df1, x='age',nbins=47, title='Гистограмма распределения возраста')
hist_age.update_layout(bargap=0.05)
hist_age.show()

Посчитаем количество мужчин и женщин и их % от общего количества

In [ ]:
all = df1['sex'].count()
men = (df1['sex'] == 'male').sum()
women = (df1['sex'] == 'female').sum()
men_share = men/all * 100
women_share = women/all * 100
print(f'Количество мужчин = {men}')
print(f'Количество женщин = {women}')
print(f'Мужчин {round(men_share,2)}%')
print(f'Женщин {round(women_share,2)}%')

Количество мужчин = 675
Количество женщин = 662
Мужчин 50.49%
Женщин 49.51%


Найдем средний ИМТ

In [ ]:
print(f"Cредний ИМТ = {round(df1['bmi'].mean(),2)}")


Cредний ИМТ = 30.66


Найдем количество курящих и некурящих

In [ ]:
print(f"Курят {(df1['smoker'] == 'yes').sum()}")
print(f"Не курят {(df1['smoker'] == 'no').sum()}")

Курят 274
Не курят 1063


Найдем у скольких людей есть дети

In [ ]:
print(f"Дети есть у {(df1['children'] > 0).sum()} человек")

Дети есть у 764 человек


Найдем количество человек по регионам

In [ ]:
region_groups = df1.groupby('region')
region_groups.size()

region
northeast    324
northwest    324
southeast    364
southwest    325
dtype: int64

Найдем среднюю выплату для мужчин и женщин

In [ ]:
sex_groups = df1.groupby('sex')['charges'].mean()
sex_groups

sex
female    12569.578844
male      13974.998864
Name: charges, dtype: float64

*   В распределении возраста имеется большое кол-во человек возрастом 18-19 лет
*   Мужчин в выборке примерно столько же, сколько и женщин.
*   Средний индекс массы тела = 30.7, что выше среднего по ВОЗ
*   Люди довольно равномерно распределены по регионам
*   Некурящих в выборке больше, чем курящих
*   Дети есть у 764 человек из 1337, что больше половины
*   Выплаты мужчинам в среднем чуть выше, чем женщинам
*   В целом данные сбалансированы, но есть аномально большое количество наблюдений 18-19 лет и у всех довольно высокий ИМТ
* Также стоит отметить очень большой разброс в выплатах, судя по стандартному отклонению, из-за чего точность модели может страдать

## 2.3

In [ ]:
max_charge = df1[df1['charges'] == df1['charges'].max()]
min_charge = df1[df1['charges'] == df1['charges'].min()]

In [ ]:
max_charge

,age,sex,bmi,children,smoker,region,charges
543,54,female,47.41,0,yes,southeast,63770.42801


In [ ]:
min_charge

,age,sex,bmi,children,smoker,region,charges
940,18,male,23.21,0,no,southeast,1121.8739


Человек с максимальной выплатой это курящая женщина 54 лет, имеющая очень высокий ИМТ, она получила более 63000. При этом самую малую выплату получил некурящий мужчина 18 лет с ИМТ в два раза меньше. Они оба не имеют детей и живут в Southeast регионе

## 2.4

In [ ]:
df1['sex'] = df1['sex'].replace({'male': 0, 'female': 1}) #меняем значения пола
df1['smoker'] = df1['smoker'].replace({'yes': 0, 'no': 1}) #меняем значения курения
df1['region'] = df1['region'].replace({'southeast': 0, 'southwest': 1, 'northeast': 2, 'northwest': 3}) #меняем значения региона

pd.set_option('future.no_silent_downcasting', True) #чтобы не было предупреждений


In [ ]:
df1.head() #проверим, применились ли изменения

,age,sex,bmi,children,smoker,region,charges
0,19,1,27.900,0,0,1,16884.92400
1,18,0,33.770,1,1,0,1725.55230
2,28,0,33.000,3,1,0,4449.46200
3,33,0,22.705,0,1,3,21984.47061
4,32,0,28.880,0,1,3,3866.85520


## 2.5

In [ ]:
X1 = df1.drop('charges', axis=1) #признаки
y1 = df1[['charges']] #целевая переменная

In [ ]:
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size = 0.3, random_state = 52) # разделили выборку на тренировочные и тестовые данные, 70% обучение, 30% тест, рандомный выбор данных для выборок

In [ ]:
model1 = LinearRegression() #инициализировали модель
model1.fit(X1_train,y1_train) #обучили модель

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [ ]:
test1_predict = model1.predict(X1_test) # Считаем выходы по тестовой выборке
train1_predict = model1.predict(X1_train) # Считаем выходы по тренировочной выборке

In [ ]:
MAE_Test1 = mean_absolute_error(test1_predict, y1_test)
MAE_Train1 = mean_absolute_error(train1_predict, y1_train)

r2_Test1 = r2_score(y1_test, test1_predict)
r2_Train1 = r2_score(y1_train, train1_predict)

RMSE_Test1 = np.sqrt(mean_squared_error(y1_test, test1_predict))
RMSE_Train1 = np.sqrt(mean_squared_error(y1_train, train1_predict))

print(f"MAE Test = {round(MAE_Test1,1)}")
print(f"MAE Train = {round(MAE_Train1,1)}")
print(f"R2 Test = {round(r2_Test1,2)}")
print(f"R2 Train = {round(r2_Train1,2)}")
print(f"RMSE Test = {round(RMSE_Test1,1)}")
print(f"RMSE Train = {round(RMSE_Train1,1)}")

MAE Test = 4422.6
MAE Train = 4099.9
R2 Test = 0.74
R2 Train = 0.75
RMSE Test = 6368.9
RMSE Train = 5911.6


Показатели MAE и RMSE довольно высокие, учитывая, что средняя выплата по страховке около 13270, но дело в том, что и стандартное отклонение составляет 12151, поэтому можно сказать, что сами данные имеют сильный разброс. На этом фоне R2 довольно высок, особенно на тренировочной выборке, что говорит о том, что модель хорошо описывает разброс данных. Но дальше нужно попробовать отбор признаков, чтобы улучшить показатели модели

## 2.6

Для начала создам копию исходника

In [ ]:
df2 = df1.copy() #копируем исходник

* Можно разделить на возрастные группы: 18-25, 26-35, 36-45, 46-55, 56-65, 65+, чтобы в каждую попало больше наблюдений

* Для BMI можно использовать категории в соответствии с ВОЗ

* Также можно добавить признак взаимодействия возраста и BMI, поскольку эти два признака могут влиять на выплату совместно, например молодой человек с низким BMI может иметь меньшую выплату, чем пожилой человек с высоким BMI

* Можно разделить курильщиков не на 0 и 1, а давать 'штраф' тому, кто курит, и умножать этот штраф на bmi, тогда курильщик с лишним весом будет иметь больший индекс, чем курильщик с обычным весом. Я проверю два варианта, когда штраф за курение увеличивает индекс в 2 раза, и в 10 раз

* Также можно перевести признак наличия детей в категориальный, так как наличие детей может влиять само по себе, вне зависимости от количества

* Курильщиков хотелось бы отмечать как 1 = курит и 0 = не курит, чтобы можно было посмотреть положительную корреляцию с выплатами

In [ ]:
df2['age_group'] = pd.cut(df2['age'], bins=[0, 25, 35, 45, 55, 65, 100], labels=[0, 1, 2, 3, 4, 5])
#0 - 18-25, 1 - 26-35, 2 - 36-45, 3 - 46-55, 4 - 56-65, 5 - 65+
df2['bmi_category'] = pd.cut(df2['bmi'], bins=[0, 18.5, 25, 30, 35, 40, 100], labels=[0, 1, 2, 3, 4, 5])
#0 - недостаточный вес, 1 - нормальный, 2 - избыточный, 3 - ожирение 1 степени, 4 - ожирение 2 степени, 5 - ожирение 3 степени
df2['age*bmi_index'] = df2['age'] * df2['bmi']
# умножаем вес на ИМТ
df2['smoker'] = 1 - df2['smoker']
# делаем так чтобы курильщик = 1, некурящий = 0
df2['smoke_penalty_10'] = (np.where(df2['smoker'] == 0, 0.1, 1))*df2['bmi']
# тут ИМТ курящего умножается на 10, а некурящего остается как был
df2['smoke_penalty_2'] = (np.where(df2['smoker'] == 0, 0.5, 1))*df2['bmi']
# тут ИМТ курящего умножается на 2, а некурящего остается как был
df2['has_children'] = (df2['children'] > 0).astype(int)
# переводим признак количества детей в категориальный: есть, нет

In [ ]:
correlation_with_charges = df2.corr()['charges'].sort_values(ascending=False)
print("Корреляция признаков с charges:")
print(correlation_with_charges)

Корреляция признаков с charges:
charges             1.000000
smoke_penalty_10    0.849597
smoke_penalty_2     0.814045
smoker              0.787234
age*bmi_index       0.334184
age                 0.298308
age_group           0.284461
bmi_category        0.204846
bmi                 0.198401
children            0.067389
has_children        0.063985
region             -0.056070
sex                -0.058044
Name: charges, dtype: float64


С выплатами лучше всего коррелируют
1) штраф за курение, умноженный на BMI, причем десятикратный
2) Сам факт курения
3) age*bmi index

Я попробовал тестировать разные наборы из этих признаков, однако лучший результат был показан тогда, когда они все работали вместе

In [ ]:
X2 = df2.drop('charges', axis=1) #берем все признаки, и старые, и новые, тк у нас не такой огромный датасет, чтобы думать о вычислительных мощностях
y2 = df2[['charges']] #целевая переменная

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size = 0.3, random_state = 69)

In [ ]:
model2 = LinearRegression() #инициализировали модель
model2.fit(X2_train,y2_train) #обучили модель

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [ ]:
test2_predict = model2.predict(X2_test) # Считаем выходы по тестовой выборке
train2_predict = model2.predict(X2_train) # Считаем выходы по тренировочной выборке

In [ ]:
MAE_Test2 = mean_absolute_error(test2_predict, y2_test)
MAE_Train2 = mean_absolute_error(train2_predict, y2_train)
r2_Test2 = r2_score(y2_test, test2_predict)
r2_Train2 = r2_score(y2_train, train2_predict)
RMSE_Test2 = np.sqrt(mean_squared_error(y2_test, test2_predict))
RMSE_Train2 = np.sqrt(mean_squared_error(y2_train, train2_predict))
print(f"MAE Test = {round(MAE_Test2,1)}")
print(f"MAE Train = {round(MAE_Train2,1)}")
print(f"R2 Test = {round(r2_Test2,2)}")
print(f"R2 Train = {round(r2_Train2,2)}")
print(f"RMSE Test = {round(RMSE_Test2,1)}")
print(f"RMSE Train = {round(RMSE_Train2,1)}")

MAE Test = 3033.0
MAE Train = 2786.7
R2 Test = 0.82
R2 Train = 0.85
RMSE Test = 5310.9
RMSE Train = 4589.4


MAE на тестовой выборке с 4400 снизилось до 3033, а R2 вырос с 0,74 до 0,82. Это конечно далеко не идеальная модель, но стало лучше, чем было